In [ ]:
import os
from pathlib import Path
# Resolve submission_partition_draft root from common launch locations.
def _resolve_partition_root():
    cwd = Path(os.getcwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "primary_script").exists() and (p / "intermediate").exists():
            return p
        candidate = p / "technical_review" / "submission_partition_draft"
        if (candidate / "primary_script").exists():
            return candidate
    raise RuntimeError("Could not locate submission_partition_draft root")
REPO_ROOT = _resolve_partition_root()


In [ ]:
#print('Scanpy version:', sc.__version__)

In [ ]:
#!pip install --upgrade scan

In [ ]:
#print('PyNNDescent version:', pynndescent.__version__)

In [ ]:
#import pynndescent

In [ ]:
#!pip install --upgrade pynndescent

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import scvi
import torch
import warnings
import leidenalg as la
from matplotlib import pyplot as plt
from matplotlib.pyplot import rc_context
from scvi.autotune import ModelTuner
from ray import tune
from scipy.io import mmwrite

In [ ]:
os.makedirs(str(REPO_ROOT / 'intermediate' / 'pbmc'), exist_ok=True)
os.makedirs(str(REPO_ROOT / 'intermediate' / 'adipose'), exist_ok=True)

In [ ]:
warnings.filterwarnings('ignore')
sc.set_figure_params(dpi=200)
plt.rcParams['figure.figsize'] = [3,3]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
os.chdir(str(REPO_ROOT / 'intermediate'))

In [ ]:
adata = sc.read_h5ad("mm_pbmc.h5ad")

In [ ]:
np.all(adata.X.data == adata.X.data.astype(np.int32))

In [ ]:
sc.pp.filter_cells(adata, min_genes = 200)
sc.pp.filter_genes(adata, min_cells = 10)
adata.var['MT'] = adata.var_names.str.startswith('MT-')
adata = adata[adata.obs.percent_mito <= 10]

In [ ]:
adata.layers['counts'] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.raw = adata

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000, subset = False, layer = 'counts', 
                            flavor = "seurat_v3", batch_key="lane")

In [ ]:
adata_var = adata.var
csv_data = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'EXCLUDE_XY_TCR_IG.csv'), header = None)

positions = np.where(adata.var_names.isin(csv_data[0]))[0]
adata.var.iloc[positions, adata.var.columns.get_loc('highly_variable')] = False
adata.var.loc[adata.var.index=="TRAV1-2", "highly_variable"] = True # force TRAV1-2 to be hvg; MAIT marker
adata.var['highly_variable'].value_counts()

In [ ]:
model_cls = scvi.model.SCVI
model_cls.setup_anndata(adata = adata, layer = "counts", batch_key = 'lane', 
                        categorical_covariate_keys=['study_id'], 
                        continuous_covariate_keys=['percent_mito', 'nCount_RNA'])
tuner = ModelTuner(model_cls)

In [ ]:
tuner.info()

In [ ]:
search_space = {
    "n_hidden": tune.choice([92, 128, 192, 256]),
    "n_latent": tune.choice([10, 20, 30, 40, 50, 60]),
    "n_layers": tune.choice([1, 2, 3]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "gene_likelihood": tune.choice(["nb", "zinb"])}

In [ ]:
results = tuner.fit(adata, metric="validation_loss",
                    resources = {'gpu': 1}, 
                    search_space = search_space,
                    num_samples = 100,
                    max_epochs = 20)

In [ ]:
best_vl = 10000
best_i = 0
for i, res in enumerate(results.results):
    vl = res.metrics['validation_loss']

    if vl < best_vl:
        best_vl = vl
        best_i = i

In [ ]:
results.results[best_i]

![parameter_tuning](/media/MPEdge16/MM137/sc/py/misc/optimal_integration_parameters_round_1.png)

In [ ]:
scvi.model.SCVI.setup_anndata(adata = adata, layer = "counts", batch_key = 'lane', 
                              categorical_covariate_keys=['study_id'], 
                              continuous_covariate_keys=['percent_mito', 'nCount_RNA'])

In [ ]:
model = scvi.model.SCVI(adata, n_hidden = 192, n_latent = 30, n_layers = 2, gene_likelihood = 'zinb')
kwargs = {'lr': 0.0020}

In [ ]:
model.train(max_epochs = 200, early_stopping = True, plan_kwargs = kwargs)

In [ ]:
y = model.history['reconstruction_loss_validation']['reconstruction_loss_validation'].min()

In [ ]:
plt.plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')
plt.plot(model.history['reconstruction_loss_validation']['reconstruction_loss_validation'], label='validation')

plt.axhline(y, c = 'k')

plt.legend()
plt.show()

In [ ]:
adata = sc.read_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'temp_pbmc.h5ad'))
#model.save('pbmc/pbmc_scvi_integration_model')
model = scvi.model.SCVI.load(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_scvi_integration_model/'), adata)
adata.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'temp_pbmc.h5ad'))

In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation()
scvi_norm_count = model.get_normalized_expression(library_size = 1e4)
adata.layers['scvi_normalized'] = scvi_norm_count

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI', random_state = 123)

In [ ]:
sc.tl.leiden(adata, resolution = 3, key_added = 'overcluster')

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_int.h5ad'))

In [ ]:
sc.pl.umap(adata, color = 'lane', size = 1, legend_fontsize = 'medium')

In [ ]:
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color = ['study_id', 'study_day'], size = 1, frameon = False)

In [ ]:
sc.set_figure_params(dpi=200)

sc.pl.umap(adata, color = ['CD3E','CD8A','CD79A','NKG7','CD14','FCGR3A','PF4','LYZ','TRAV1-2','TRGV9','MKI67','CD34','SELL','LEF1','BANK1','CD4'], size = 2, frameon = False, layer = 'scvi_normalized')

In [ ]:
plt.rcParams['figure.figsize'] = [7, 7]
sc.pl.umap(adata, color = ['JCHAIN','overcluster'], legend_loc = 'on data', size = 2, layer = 'scvi_normalized', legend_fontsize = 'small')

In [ ]:
sc.set_figure_params(dpi=100)
sc.pl.dotplot(adata, ['IL7R', 'TRAC', 'LEF1', 'CD3E', 'CD8A', 'CD8B', 'LINC02446', 'CD19', 'MS4A1', 'CD79A', 'BANK1', 'IGHA2', 'TNFRSF17', 'HRASLS2', 'MKI67', 'NKG7', 'XCL1', 'KLRD1', 'KLRF1', 'KLRB1', 'TRAV1-2', 'SLC4A10', 'FCGR3A', 'CD14', 'NEAT1', 'LYZ', 'S100A9', 'ITM2C','JCHAIN','PLD4','SERPINF1', 'PF4', 'HIST1H2AC', 'GP9', 'KIT', 'SOX4','CD34','FOXP3'], groupby = 'overcluster', swap_axes = True,
              use_raw = True, standard_scale = 'var', dendrogram = False)

In [ ]:
#uclus = max(adata.obs['overcluster'].astype(int))
#for i in range(uclus+1):
#    print("'" + str(i) + "'" + ": " + "'" + "',")

In [ ]:
merge_type_ref = {
    '0': 'TNK',
    '1': 'B',
    '2': 'TNK',
    '3': 'TNK',
    '4': 'TNK',
    '5': 'Myeloid',
    '6': 'Myeloid',
    '7': 'TNK',
    '8': 'Myeloid',
    '9': 'TNK',
    '10': 'TNK',
    '11': 'TNK',
    '12': 'TNK',
    '13': 'B',
    '14': 'TNK',
    '15': 'TNK',
    '16': 'TNK',
    '17': 'TNK',
    '18': 'TNK',
    '19': 'Myeloid',
    '20': 'B',
    '21': 'Myeloid',
    '22': 'TNK',
    '23': 'Myeloid',
    '24': 'Platelet',
    '25': 'TNK',
    '26': 'Myeloid',
    '27': 'TNK',
    '28': 'Myeloid',
    '29': 'TNK',
    '30': 'B',
    '31': 'B',
    '32': 'HSPC',
    '33': 'B',
    '34': 'Myeloid',
    '35': 'TNK',
    '36': 'TNK',
    '37': 'TNK'
}

In [ ]:
adata.obs['merged_type'] = adata.obs['overcluster'].map(merge_type_ref)
adata.obs

In [ ]:
plt.rcParams['figure.figsize'] = [7, 7]
sc.pl.umap(adata, color = ['merged_type'], legend_loc = 'on data', size = 2, layer = 'scvi_normalized', legend_fontsize = 'small')

In [ ]:
adata_tnk = adata[adata.obs['merged_type']=='TNK']
adata_b = adata[adata.obs['merged_type']=='B']
adata_myeloid = adata[adata.obs['merged_type']=='Myeloid']
adata_platelet = adata[adata.obs['merged_type']=='Platelet']
adata_hspc = adata[adata.obs['merged_type']=='HSPC']

In [ ]:
adata_tnk.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_subset_tnk.h5ad'))
adata_b.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_subset_b.h5ad'))
adata_myeloid.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_subset_myeloid.h5ad'))
adata_platelet.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_subset_platelet.h5ad'))
adata_hspc.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_subset_hspc.h5ad'))

In [ ]:
rna_counts = adata_platelet.layers['counts']
adata_obs = adata_platelet.obs
adata_var = adata_platelet.var

mmwrite(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/adata_pbmc_platelet_counts.mtx'), rna_counts)
adata_obs.to_csv(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/adata_pbmc_platelet_obs.csv'))
adata_var.to_csv(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/adata_pbmc_platelet_var.csv'))